# Soluzioni — Lezione 2: I dati e le parole
[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/main/ml-humanities/solutions/02_dati_e_parole_alberi_soluzioni.ipynb)

In [1]:
# ⚙️ CELLA DI PREPARAZIONE — eseguitela per prima (Shift+Invio) e non preoccupatevi del codice.
# Cerca la cartella "data" del corso; se non la trova (per esempio su Google Colab),
# scarica il materiale del corso da GitHub. Funziona anche senza internet, se avete la cartella.
from pathlib import Path
import subprocess

def trova_dati():
    for p in [Path("../data"), Path("data"), Path("ml-humanities/data"), Path("yuribizzoni/ml-humanities/data")]:
        if (p / "lessico_felicita.csv").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuri-bizzoni/yuribizzoni"], check=True)
    return Path("yuribizzoni/ml-humanities/data")

DATI = trova_dati()
print("Dati trovati in:", DATI.resolve())

Dati trovati in: /home/user/yuribizzoni/ml-humanities/data


In [2]:
import re
from collections import Counter
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def parole(testo):
    """Spezza un testo in parole minuscole."""
    return re.findall(r"\w+", testo.lower())

def leggi_testo(nome):
    """Legge un testo dalla cartella data/testi, saltando le righe di intestazione che iniziano con #."""
    righe = (DATI / "testi" / nome).read_text(encoding="utf-8").splitlines()
    return "\n".join(r for r in righe if not r.startswith("#"))

STOPWORDS = set()
for riga in (DATI / "stopwords_it.txt").read_text(encoding="utf-8").splitlines():
    if not riga.startswith("#"):
        STOPWORDS.update(riga.split())

famiglie = pd.read_csv(DATI / "famiglie_felicita_it.csv")
FAMIGLIA_DI = dict(zip(famiglie["forma"], famiglie["famiglia"]))   # forma → famiglia, es. "lieta" → "letizia"
FAMIGLIE = list(dict.fromkeys(famiglie["famiglia"]))

RISULTATI = Path("risultati")
RISULTATI.mkdir(exist_ok=True)      # qui salveremo i nostri CSV

## Esercizio 1 — Le parole di una poesia

In [3]:
canti = pd.read_csv(DATI / "leopardi_canti.csv")
silvia = canti.loc[canti["titolo"] == "A Silvia", "testo"].iloc[0]
Counter(p for p in parole(silvia) if p not in STOPWORDS and len(p) > 2).most_common(10)

[('allor', 3),
 ('silvia', 2),
 ('tempo', 2),
 ('vita', 2),
 ('vie', 2),
 ('opre', 2),
 ('speme', 2),
 ('natura', 2),
 ('anni', 2),
 ('dolce', 2)]

**Commento.** In una poesia breve le frequenze sono piccole (2–3 occorrenze): pochi dati, molta prudenza.
Emergono comunque i nuclei della poesia: il tempo, la speranza, la giovinezza, il ricordo.

## Esercizio 2 — Un CSV della felicità leopardiana

In [4]:
righe = []
for _, poesia in canti.iterrows():
    riga = {"titolo": poesia["titolo"], "n_parole": len(parole(poesia["testo"]))}
    for f in FAMIGLIE:
        riga[f] = 0
    for p in parole(poesia["testo"]):
        if p in FAMIGLIA_DI:
            riga[FAMIGLIA_DI[p]] += 1
    righe.append(riga)

leo = pd.DataFrame(righe)
leo.to_csv(RISULTATI / "leopardi_felicita.csv", index=False)
print("Salvato:", RISULTATI / "leopardi_felicita.csv")
leo[FAMIGLIE].sum().sort_values(ascending=False)

Salvato: risultati/leopardi_felicita.csv


felicità       33
beatitudine    24
letizia        22
contentezza    13
gioia          10
allegria        6
dtype: int64

**Commento.** In Leopardi la famiglia più frequente è quella della **felicità** in senso stretto (*felice, felici*), spesso negata
o rimpianta; seguono *beatitudine* (*beato, beata*: spesso nel senso di "fortunato", come nel *Canto notturno*) e la *letizia*
letteraria (*lieto, lieta*). In Pinocchio domina invece la **contentezza** quotidiana.
Registri diversi, felicità diverse.

## Esercizio 3 — Cinque autori

In [5]:
import numpy as np

def caratteristiche(testo):
    """Trasforma un testo in una riga di numeri (caratteristiche stilistiche)."""
    p = parole(testo)
    frasi = [f for f in re.split(r"[.!?]+", testo) if f.strip()]
    n = len(p)
    return {
        "lung_media_parola": np.mean([len(x) for x in p]),
        "parole_per_frase": n / len(frasi),
        "esclamativi_100": testo.count("!") / n * 100,
        "trattini_100": testo.count("—") / n * 100,        # il trattino lungo introduce i dialoghi
        "punto_e_virgola_100": testo.count(";") / n * 100,
        "funzionali_100": sum(x in STOPWORDS for x in p) / n * 100,
        "felici_1000": sum(x in FAMIGLIA_DI for x in p) / n * 1000,
    }

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

passaggi = pd.read_csv(DATI / "passaggi_autori.csv")
X_tutti = pd.DataFrame([caratteristiche(t) for t in passaggi["testo"]])
y = passaggi["autore"]
X_train, X_test, y_train, y_test = train_test_split(X_tutti, y, test_size=0.3, random_state=0, stratify=y)

for profondita in [3, 10]:
    albero = DecisionTreeClassifier(max_depth=profondita, random_state=0).fit(X_train, y_train)
    print(f"max_depth={profondita}: train {albero.score(X_train, y_train):.2f}, test {albero.score(X_test, y_test):.2f}")

albero3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
pd.Series(albero3.feature_importances_, index=X_tutti.columns).sort_values(ascending=False).round(3)

max_depth=3: train 0.55, test 0.42
max_depth=10: train 0.95, test 0.45


trattini_100           0.567
punto_e_virgola_100    0.225
funzionali_100         0.140
lung_media_parola      0.067
parole_per_frase       0.000
esclamativi_100        0.000
felici_1000            0.000
dtype: float64

**Commento.** Con cinque autori il compito è molto più difficile: con il caso puro si indovinerebbe il 20%.
Con `max_depth=10` l'albero arriva al 95% sui dati di addestramento, ma sul test resta sotto il 50%:
è l'**overfitting**, l'imparare a memoria invece di generalizzare. Le caratteristiche più usate sono di punteggiatura e
lunghezza delle frasi: segnali di **stile** (e di edizione!), non di contenuto.